# MindsOS — intelligence on device, loaded on demand, open to audit

*A reproducible walkthrough. Everything below runs on **CPU** — the same code is tested on a 2012 Mac Mini. Run it top-to-bottom at the MindsOS repo root in the project venv.*

## 1 · The problem

The dominant path to AI buys capability with **scale**: cloud, GPUs, one enormous model holding all knowledge at once, its reasoning fused into weights you cannot inspect. For anyone who must **control, audit, and run intelligence on their own hardware** — a regulated operator, an edge or offline site, a data-sovereign or high-assurance setting — that path is effectively closed. You get the answer or you don't; you cannot see why, run it locally, or bound what it knows.

## 2 · What MindsOS is (and what it is not)

MindsOS is an **architecture designed to produce intelligence** from a society of independent, typed, composable, inspectable parts. It is **not** an integration of existing tools — the reuse of known techniques is a *consequence* of the design, not its motivation.

**Honest bound, stated up front.** The properties you'll see below — data-efficiency, low compute, learning without forgetting, transfer — are the *generic dividend* of any well-designed compositional architecture. Few-shot class-incremental learning, library learning, and neuro-symbolic continual learning already obtain them. So these demonstrations show the architecture **works** — they are **not** a claim to a novel mechanism.

**What is distinctive is the stance:** MindsOS runs **on device**, holds **local** knowledge, **loads only what a task needs on demand**, keeps everything on **one inspectable and auditable** representation, and **grows by being taught** rather than retrained. That combination — intelligence you can run locally *and* audit *and* extend by teaching — is what the scale-and-cloud path does not offer.

## 3 · Who it is for

Not "everyone who wants more with less" — that is the pitch of every distillation and small-model vendor, and it is table stakes. MindsOS is for whoever needs efficiency **together with control, audit, and locality**: regulated industries, edge and offline deployments, data-sovereign or high-assurance operators. Efficiency alone they can buy anywhere; efficiency they can **inspect, audit, and run on their own hardware**, almost no one offers.

## 4 · What follows

First, that it **works, on CPU** — it composes a procedure no one wrote, runs it, and refuses honestly. Then two separate claims the on-device tests support: the **hardware envelope** (runs on a 2012 Mac Mini) and **compute efficiency** (FLOPs-at-matched-competence). Then why, *in this architecture*, loading knowledge on demand and learning a new capability are the same act. Then an honest map of what is built vs not, and the ask.

In [ ]:
# Visual: render a composed pipeline (steps + typed dataflow edges)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
def _short(iri): return iri.split(":")[-1].split(".")[-1]
def draw_pipeline(pipeline, title=""):
    steps, starts = list(pipeline.steps), list(pipeline.start_datastates)
    fig, ax = plt.subplots(figsize=(max(8, 2.8*(len(steps)+1)), 2.9)); ax.axis("off")
    x0, y0, dx = 0.7, 0.5, 2.8; node = {}
    ax.add_patch(FancyBboxPatch((x0-0.55, y0-0.22), 1.1, 0.44,
        boxstyle="round,pad=0.02,rounding_size=0.08", fc="#f4f1e8", ec="#7a8ca0", lw=1.4))
    ax.text(x0, y0, ("start\n"+", ".join(_short(s) for s in starts)) if starts else "start",
            ha="center", va="center", fontsize=8, color="#333")
    for i, s in enumerate(steps):
        x = x0+dx*(i+1); node[i] = (x, y0)
        ax.add_patch(FancyBboxPatch((x-0.62, y0-0.26), 1.24, 0.52,
            boxstyle="round,pad=0.02,rounding_size=0.1", fc="#2f5d8a", ec="#1c3a56", lw=1.5))
        ax.text(x, y0, _short(s.capacity_iri), ha="center", va="center",
                fontsize=9, color="white", fontweight="bold")
    xy = lambda idx: (x0, y0) if idx == -1 else node[idx]
    for e in pipeline.edges:
        (sx, sy), (tx, ty) = xy(e.producer), node[e.consumer]
        ax.add_patch(FancyArrowPatch((sx+0.62, sy), (tx-0.62, ty),
            arrowstyle="-|>", mutation_scale=13, lw=1.3, color="#4a4a4a"))
        ax.text((sx+tx)/2, sy+0.26, _short(e.datastate), ha="center", va="bottom",
                fontsize=7, color="#2f5d8a", style="italic")
    ax.set_xlim(-0.2, x0+dx*(len(steps)+1)+0.4); ax.set_ylim(-0.2, 1.2)
    if title: ax.set_title(title, fontsize=10, color="#222", loc="left")
    plt.tight_layout(); plt.show()

## 5 · Teach the primitives (not the procedure)

Register five typed **DataStates** (the vocabulary) and four single-purpose **Capacities** (the operations). We never wire them into a pipeline — composition is the system's job.

In [ ]:
from mindsos_capacity import (
    Capacity, CapacityLayer, DataState, ShapeDescriptor,
    INPUT_GROUP_ALL_REQUIRED, CATEGORY_PERCEPTION, CATEGORY_COMPREHENSION, CATEGORY_DECISION,
)
from mindsos_capacity.pipeline import find_pipeline
from mindsos_capacity.exceptions import PipelineNotFoundError
from mindsos_capacity.needs_input import NeedsInput
from mindsos_capacity.runtime import invoke
from mindsos_intelligence.pipeline_execution import execute_pipeline

def IRI(s): return f"datastate:t.{s}"
RAW, PARSED, NORMAL, CONDITION, ACTION, DIAGNOSIS = map(
    IRI, ("raw_signal","parsed_signal","normal_signal","condition","action","diagnosis"))
CONDITIONS = {"pressure_high": "vent", "pressure_low": "seal", "nominal": "hold"}

def _parse(**kw):      return {PARSED: str(kw[RAW]).strip().lower()}
def _normalize(**kw):  return {NORMAL: kw[PARSED].replace(" ", "_")}
def _classify(**kw):
    v = kw[NORMAL]
    if v not in CONDITIONS:
        return NeedsInput(question=f"Unrecognized reading {v!r}; which condition is this?",
                          missing=CONDITION, choices={c: c for c in CONDITIONS})
    return {CONDITION: v}
def _recommend(**kw):  return {ACTION: CONDITIONS[kw[CONDITION]]}

def _ds(s):
    n = f"t.{s}"; return DataState(name=n, shape=ShapeDescriptor.scalar("str", opaque_tag=n))
def _cap(name, cat, ins, outs, impl):
    return Capacity(name=name, category=cat, inputs=tuple(ins), outputs=tuple(outs),
                    input_group=INPUT_GROUP_ALL_REQUIRED, implementation=impl)

cl = CapacityLayer(categories=(CATEGORY_PERCEPTION, CATEGORY_COMPREHENSION, CATEGORY_DECISION))
for s in ("raw_signal","parsed_signal","normal_signal","condition","action","diagnosis"):
    cl.register_datastate(_ds(s), allow_new_realm=True)
cl.register_capacity(_cap("parse",     CATEGORY_PERCEPTION,    [RAW],       [PARSED],    _parse))
cl.register_capacity(_cap("normalize", CATEGORY_COMPREHENSION, [PARSED],    [NORMAL],    _normalize))
cl.register_capacity(_cap("classify",  CATEGORY_DECISION,      [NORMAL],    [CONDITION], _classify))
cl.register_capacity(_cap("recommend", CATEGORY_DECISION,      [CONDITION], [ACTION],    _recommend))

class Dispatcher:
    def __init__(self, cl): self.cl = cl
    def dispatch(self, ci, inputs, *, cancel_token=None, task_id=None, step_id=None):
        return invoke(self.cl.get_declaration(ci), inputs, task_id=task_id, step_id=step_id)
print("Registered 4 capacities + 6 datastates. No pipeline was wired.")

## 6 · It composes a procedure nobody wrote — and runs it, on CPU

In [ ]:
pipe = find_pipeline(cl, start_datastate=RAW, target_datastate=ACTION)
print("composed:", " -> ".join(s.capacity_iri.split(":")[-1] for s in pipe.steps))
draw_pipeline(pipe, "Composed by search from raw_signal → action (4 steps, none hand-wired)")
res = execute_pipeline(Dispatcher(cl), pipe, {RAW: "  Pressure High "}, task_id="demo-1")
print("ran:", res.success, "| action =", res.outputs.get(ACTION))

## 7 · It refuses instead of guessing — at the unit, and at the whole task

In [ ]:
# unit-level: an out-of-vocabulary reading -> typed request, not a fabricated answer
r2 = execute_pipeline(Dispatcher(cl), pipe, {RAW: "gremlins"}, task_id="demo-2")
print("unit refusal:", r2.needs_input.question if r2.needs_input else r2.outputs.get(ACTION))

# task-level: a goal with no route -> honest "no pipeline", not a crash or a guess
try:
    find_pipeline(cl, start_datastate=RAW, target_datastate=DIAGNOSIS)
except PipelineNotFoundError as exc:
    print("no route:", exc)

## 8 · Two claims the on-device tests support (keep them separate)

**(a) The hardware envelope.** The same code runs on a **2012 Mac Mini, CPU-only**, holding only the knowledge a task needs. A cloud-scale model cannot deploy into that envelope at all — different kind of claim from raw speed.

**(b) Compute efficiency.** The honest metric is **FLOPs-to-competence**: FLOPs (and wall-clock) to reach the *same task result*, on matched hardware. Two disciplines make the comparison fair and, in fact, favour MindsOS honestly:
- compare **at matched competence** (report accuracy, or it's a trade-off point, not a win);
- count the baseline's **total** FLOPs including **pretraining** — that is where a large model's real cost lives.

The cell below is a **scaffold**: it times this CPU run as an illustration and marks where the real, matched-competence FLOPs comparison against a neural baseline goes. Fill it in from your Mac Mini run — do not present illustrative timings as the result.

In [ ]:
import time
t0 = time.perf_counter()
_ = execute_pipeline(Dispatcher(cl), pipe, {RAW: "Pressure High"}, task_id="compute-demo")
mindsos_walltime_s = time.perf_counter() - t0
print(f"MindsOS compose+run wall-clock (illustrative, CPU): {mindsos_walltime_s*1000:.2f} ms")

# TODO (real comparison — run on the Mac Mini, at MATCHED competence):
#   MINDSOS_FLOPS   = <profile the MindsOS run>
#   BASELINE_FLOPS  = <neural baseline: inference FLOPs + amortized PRETRAINING FLOPs>
#   Report accuracy for BOTH at the same task; the claim is X_flops < Y_flops AT equal competence.
print("TODO: fill MINDSOS_FLOPS vs BASELINE_FLOPS (incl. baseline pretraining) at matched competence.")

## 9 · Here, loading knowledge on demand *is* incremental learning

Because MindsOS knowledge lives as **independent, typed units**, two things that are separate — and mutually interfering — in a fused-weight model become the **same kind of act on one representation**:

- **load on demand** = select the units a task needs into its working model;
- **learn something new** = add a unit.

Both are add/select operations on discrete, independent objects, so neither disturbs the rest. A monolithic network cannot do either cleanly: all knowledge is entangled in shared weights, so loading isn't selective and learning perturbs everything (catastrophic forgetting). *(Precise version: knowledge **instances** are independent; **capabilities** are composed from **shared** primitives — independence at one layer, reuse at another. That is what lets learning be non-destructive while still transferring.)*

## 10 · What is built, and what is not

**Built and running (everything above executed on it, on CPU):** the metagraph substrate; typed DataStates and Capacities; on-demand pipeline composition by type-directed search; end-to-end execution; capacity-level honest refusal; replanning; local, load-on-demand knowledge.

**Designed, not yet built (named, not hidden):** the learning apparatus (audited learning, dreaming, promotion); the data-efficiency / non-destructive-learning **measurements** (the four-axis study is designed, not run); the FLOPs-at-matched-competence comparison against a real baseline; store-wide provenance to raw input (today a creation stamp); pipeline-not-found as a first-class verdict (today an exception handled gracefully); executing sound multi-input fan-in.

MindsOS is **started, not finished.** This is shown to open a door, not to claim a finished product.

## 11 · The ask

Not more architecture — **validation and reach**:

- **compute and hands** to run the four-axis study properly (data / compute-at-matched-competence / no-forgetting / transfer), and the on-device FLOPs comparison, against strong baselines;
- a **research collaborator** to co-run that study (a natural Mitacs project);
- an **introduction to a control-/audit-/locality-bound operator** — regulated, edge, offline, or sovereign — where an inspectable, on-device, teachable mind is worth more than a bigger black box.

The substrate is real and runs on a 13-year-old laptop. The ask is help proving — and deploying — the parts that aren't finished.